# SHEIN 订单利润分析

本 Notebook 基于已导出的 Parquet 数据分析最近同步订单利润。成本规则来自人工输入：

- 10pcs：货品成本 1.87，打包费 0.90
- 20pcs：货品成本 2.69，打包费 0.90
- 50pcs：货品成本 6.21，打包费 0.90
- 30pcs = 10 + 20，打包费 0.90 + 0.50 续件
- 100pcs = 50 * 2，打包费 0.90 + 0.50 续件

利润口径：非退货订单 `利润 = goods_estimatedIncome + sellerShippingFee按重量分摊 - totalPerformanceServiceCharge按重量分摊 - 货品成本 - 打包费`。订单级运费收入和履约物流费优先按商品 `goodsWeight` 分摊；缺重量时按收入占比分摊，再缺则平均分摊。订单状态 1/2/6 的收入和利润从 PnL 剔除；订单状态 7 仅在 `totalPerformanceServiceCharge > 0` 时计入，否则从 PnL 剔除；订单状态 8/9 的原始收入计入售后成本 `after_sales_cost_usd`；退货订单仍计入货品成本和打包费，但有效收入、有效运费、有效履约费和利润均按 0 进入 PnL；原始未调整金额保留在 `original_*` 字段。

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/home/ubuntu/shein-api-manager')
items = pd.read_parquet(BASE_DIR / 'exports/shein_order_items_profit.parquet')
orders = pd.read_parquet(BASE_DIR / 'exports/shein_orders_profit_summary.parquet')

items.shape, orders.shape

In [ ]:
summary = pd.DataFrame([{
    '订单数': orders['order_no'].nunique(),
    '商品行数': len(items),
    '总收入_美元': items['gross_revenue_allocated_usd'].sum(),
    '货品成本_美元': items['product_cost_rule_usd'].sum(),
    '打包费_美元': items['packaging_fee_rule_usd'].sum(),
    '内部成本_美元': items['internal_cost_usd'].sum(),
    '利润_美元': items['profit_usd'].sum(),
    '利润率': items['profit_usd'].sum() / items['gross_revenue_allocated_usd'].sum(),
    '成本规则未匹配行数': (~items['cost_rule_matched']).sum(),
}])
summary

In [ ]:
by_pcs = (items.groupby('pcs', dropna=False)
    .agg(商品行数=('goods_id','count'),
         订单数=('order_no','nunique'),
         收入=('gross_revenue_allocated_usd','sum'),
         货品成本=('product_cost_rule_usd','sum'),
         打包费=('packaging_fee_rule_usd','sum'),
         利润=('profit_usd','sum'))
    .reset_index())
by_pcs['利润率'] = by_pcs['利润'] / by_pcs['收入']
by_pcs.sort_values('利润', ascending=False)

In [ ]:
by_attr = (items.groupby(['pcs','sku_attr_us'], dropna=False)
    .agg(商品行数=('goods_id','count'),
         订单数=('order_no','nunique'),
         收入=('gross_revenue_allocated_usd','sum'),
         内部成本=('internal_cost_usd','sum'),
         利润=('profit_usd','sum'))
    .reset_index())
by_attr['利润率'] = by_attr['利润'] / by_attr['收入']
by_attr.sort_values('利润', ascending=False).head(30)

In [ ]:
by_state = (items.groupby(['country','province'], dropna=False)
    .agg(订单数=('order_no','nunique'),
         商品行数=('goods_id','count'),
         收入=('gross_revenue_allocated_usd','sum'),
         利润=('profit_usd','sum'))
    .reset_index())
by_state['利润率'] = by_state['利润'] / by_state['收入']
by_state.sort_values('利润', ascending=False).head(30)

In [ ]:
unmatched = items.loc[~items['cost_rule_matched'], [
    'order_no','sku_attr_us','goods_title','seller_sku','gross_revenue_allocated_usd'
]]
unmatched.head(50)

In [ ]:
top_loss = items.sort_values('profit_usd').loc[:, [
    'order_no','order_created_at','sku_attr_us','pcs','gross_revenue_allocated_usd',
    'product_cost_rule_usd','packaging_fee_rule_usd','profit_usd','profit_margin'
]].head(30)
top_loss